### 1. Cargar base de datos DIGITS y entrenar un modelo de regresión logística. 

* Asuma que solo conoce las etiquetas de las primeras 50 imágenes de entrenamiento. 
* Use estos 50 ejemplos para entrenar el modelo de regresión logística. 
* Calcule su exactitud en un conjunto de prueba de 25% (asume que estan etiquetados). 

In [1]:
SEED = 30
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import numpy as np

np.random.seed(SEED)
X_digits, y_digits = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X_digits, y_digits)

n_labeled = 50
log_reg = LogisticRegression(solver = 'liblinear', multi_class = 'ovr')
log_reg.fit(X_train[:n_labeled], y_train[:n_labeled])
print('Exactitud de prueba entrenando con solo 50 ejemplos:', 
      log_reg.score(X_test, y_test))

Exactitud de prueba entrenando con solo 50 ejemplos: 0.8377777777777777


/home/edgarrt/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


### 2. Use k-means para agrupar todo el conjunto de entrenamiento en 50 centros

Visualice las imágenes que representan a los 50 centros, esto es, las 50 imágenes del conjunto de entrenamiento que estén más cerca a uno de los 50 centros. 

In [ ]:
from sklearn.cluster import KMeans
import numpy as nup
import matplotlib.pyplot as plt

k = 50
np.random.seed(SEED)
kmeans = KMeans(n_clusters=k)
X_digits_dist = kmeans.fit_transform(X_train)
representative_digit_idx = np.argmin(X_digits_dist, axis=0)   # Calcula la ubicación de la imagen más cercana al centro k
X_representative_digits = X_train[representative_digit_idx]   # Extrae estas imagenes representativas de los centros

plt.figure()
plt.title('Imágenes representativas de los 50 centros')
i = 1
for x in X_representative_digits:
  plt.subplot(5,10,i)
  plt.imshow(x.reshape(8,8), cmap='gray')
  plt.axis('off')
  i += 1
plt.show()

### 3. Entrene un modelo de regresión logística para estos 50 imágenes representativas. 

* Es necesario que usted etiquete manualmente estas 50 imágenes representativas, ya que asumimos que solo conocemos 50 etiquetas. 
* Calcule su exactitud en el conjunto de prueba  
* Si le da un resultado diferente al del instructor es porque etiquetó de manera diferente. 

In [ ]:
y_representative_digits = np.array([1,0,9,2,7,7,6,4,5,1,
                                    3,7,3,5,4,5,8,2,0,9,
                                    1,2,0,3,4,1,9,7,2,6,
                                    8,9,8,6,7,6,4,7,5,2,
                                    8,9,4,3,2,2,2,1,3,3])

np.random.seed(SEED)
log_reg = LogisticRegression(solver = 'liblinear', multi_class = 'ovr')
log_reg.fit(X_representative_digits, y_representative_digits)

print('Exactitud de prueba entrenando con las 50 imágenes representativas:', 
      log_reg.score(X_test, y_test))


### 4. Propagación de etiquetas (Label propagation) al x% de los datos más cercanos a los centros

* Use el modelo de kmeans para etiquetar el x% del conjunto de entrenamiento más cercano a los 50 centros 
* Use estas etiquetas para entrenar el modelo de regresión logistica. 
* Calcule su exactitud en el conjunto de prueba 
* Pruebe varios porcentajes y halle cuál produce el mejor resultado


In [ ]:
np.random.seed(SEED)

percentile_closest = 100

y_train_propagated = np.empty(len(X_train), dtype=np.int32)
for i in range(k):
  y_train_propagated[kmeans.labels_==i] = y_representative_digits[i]

X_cluster_dist = X_digits_dist[np.arange(len(X_train)), kmeans.labels_]
for i in range(k):
  in_cluster = (kmeans.labels_ == i)
  cluster_dist = X_cluster_dist[in_cluster]
  cutoff_distance = np.percentile(cluster_dist, percentile_closest)
  above_cutoff = (X_cluster_dist > cutoff_distance)
  X_cluster_dist[in_cluster & above_cutoff] = -1
  
partially_propagated = (X_cluster_dist != -1)
X_train_partially_propagated = X_train[partially_propagated]
y_train_partially_propagated = y_train_propagated[partially_propagated]
log_reg = LogisticRegression(solver = 'liblinear', multi_class = 'ovr')
log_reg.fit(X_train_partially_propagated, y_train_partially_propagated)

print('Exactitud de clasificación entrenado con el',percentile_closest, 
        '% más cercano a los centros:', log_reg.score(X_test, y_test))